# Approach 3 — Soft-Gated Mixture of Experts with a Global Fallback

**Fake-news detection · FineFake and WELFake**

This notebook implements and evaluates the final system of the project. It is
self-contained: run the cells top to bottom and it reproduces every number in
`results/moe_finefake.csv` and `results/moe_welfake.csv`.

---

## Why this architecture

**Approach 1** trains one global classifier over the whole corpus.

**Approach 2** clusters the corpus and routes each article to the single classifier of
its nearest cluster. That design loses accuracy, and the reason is a trade-off with two
opposing terms:

| Term | Effect | Sign |
|---|---|---|
| **Specialisation** | a focused model fits its own slice better | + |
| **Starvation** | each model now trains on a fraction of the data | − |

On FineFake these measure **+0.012** and **−0.017**, so the net is negative. Approach 2
also has a second failure mode: an article sitting between two clusters is assigned to
exactly one of them, essentially by a coin flip, and is then scored by a specialist that
may never have seen anything like it.

**Approach 3 (this notebook)** attacks both problems with three changes:

1. **Heterogeneous experts** — each cluster picks its own classifier family
   (Logistic Regression / Complement NB / SGD / calibrated LinearSVC) by validation score.
2. **Soft gating** — membership is a softmax over distance to *every* centroid, so
   boundary articles are scored by several experts in proportion.
3. **Global-fallback blending** — the global model is always mixed back in:
   $$p = \alpha \cdot p_{\text{global}} + (1-\alpha)\sum_c w_c\, p_c$$
   which is the direct antidote to starvation.

**Methodological guard-rails.** The global baseline also gets best-family selection, and
*every* system gets its decision threshold tuned, so nothing wins merely by being tuned.
`S6` is a deliberate control — a plain ensemble with no clustering — which tells us whether
any gain is really about routing or just about averaging models.

## 1. Setup

In [1]:
# Standard scientific stack + scikit-learn. No GPU or deep-learning dependency:
# the whole system is linear models over sparse TF-IDF, which keeps it reproducible
# and cheap enough to retrain in a couple of minutes.
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.calibration import CalibratedClassifierCV
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             matthews_corrcoef, precision_score, recall_score,
                             roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer
from sklearn.svm import LinearSVC

SEED = 42                      # one seed everywhere, so runs are comparable
FAKE, REAL = 0, 1              # repo-wide label convention
np.random.seed(SEED)

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))
print("repo root:", REPO)

repo root: /home/claude/repo_fresh


## 2. Data

Two corpora, both de-duplicated **before** splitting. That matters more than it sounds:
FineFake contains 2,362 exactly duplicated rows, and leaving them in leaks 12.5% of the
test split into training, inflating every score.

A second trap: **WELFake ships its labels inverted** relative to FineFake and ISOT.
We verified this directly — 21,266 of its label-0 articles carry a `(Reuters)` dateline
against 16 of its label-1 articles — so label 0 is *real* there. We flip it onto the
repo convention (`fake=0, real=1`). macro-F1 is symmetric so this does not change
single-corpus scores, but it silently destroys any cross-corpus experiment.

In [2]:
from src import data as D     # repo loader: handles de-duplication and schema

WELFAKE_CSV = Path("/mnt/user-data/uploads/Downloads/WELFake_Dataset.csv")

def load_welfake() -> pd.DataFrame:
    '''WELFake normalised onto the repo schema, with labels flipped to fake=0/real=1.'''
    df = pd.read_csv(WELFAKE_CSV)
    df["title"] = df["title"].fillna("")
    df["text"] = df["text"].fillna("")
    df["text"] = (df["title"] + ". " + df["text"]).str.strip()
    df = df[df["text"].str.len() > 0].dropna(subset=["label"])
    df = df.drop_duplicates(subset=["text"])          # 11.7% of rows are duplicates
    df["label"] = 1 - df["label"].astype(int)         # <-- the polarity fix
    df["topic"] = "unknown"
    df["platform"] = "welfake"
    df["n_words"] = df["text"].str.split().str.len().astype(int)
    return df.reset_index(drop=True)[D.SCHEMA]


DATASET = "finefake"          # switch to "welfake" to reproduce the other table
df = D.load_finefake() if DATASET == "finefake" else load_welfake()
print(f"{DATASET}: {len(df):,} rows after de-duplication")
print(df["label"].value_counts().rename({FAKE: "fake", REAL: "real"}).to_dict())

finefake: 15,474 rows after de-duplication
{'fake': 8155, 'real': 7319}


### Train / validation / test

70 / 15 / 15, stratified on the label. The **validation split does all the tuning**
(cluster count, gate temperature, blend weight, decision threshold) and the **test split
is touched exactly once**, at the very end. Tuning on test would make every number below
meaningless.

In [3]:
tr, tmp = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)
va, te  = train_test_split(tmp, test_size=0.50, stratify=tmp["label"], random_state=SEED)
tr, va, te = (x.reset_index(drop=True) for x in (tr, va, te))
y_tr, y_va, y_te = (x["label"].to_numpy() for x in (tr, va, te))

print(f"train {len(tr):,}   validation {len(va):,}   test {len(te):,}")

train 10,831   validation 2,321   test 2,322


## 3. Features

Two representations from the same text, doing two different jobs:

* **Sparse TF-IDF** feeds the *classifiers*. Each dimension is a word, so coefficients
  stay inspectable.
* **Dense SVD-256, L2-normalised** feeds the *clustering and the gate*. Compact, and
  Euclidean distance in this space behaves like topical similarity.

The vectoriser is fit on **training rows only** — fitting it on everything would leak
test vocabulary statistics into the model.

In [4]:
vec = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=3,
                      max_features=100_000, sublinear_tf=True,
                      strip_accents="unicode")
X_tr = vec.fit_transform(tr["text"])      # fit on TRAIN only
X_va = vec.transform(va["text"])
X_te = vec.transform(te["text"])

svd = make_pipeline(TruncatedSVD(256, random_state=SEED), Normalizer(copy=False))
D_tr = svd.fit_transform(X_tr)
D_va = svd.transform(X_va)
D_te = svd.transform(X_te)

print("sparse TF-IDF:", X_tr.shape, " dense SVD:", D_tr.shape)

sparse TF-IDF: (10831, 100000)  dense SVD: (10831, 256)


## 4. The expert zoo

Four classifier families with different inductive biases. Logistic regression is the
reliable linear baseline; Complement NB is strong on small, skewed text samples (which is
exactly what a thin cluster is); SGD with modified-Huber loss gives probabilities with a
different regularisation path; LinearSVC is usually the strongest margin classifier on
TF-IDF but needs calibration to emit probabilities, which the blend requires.

In [5]:
def expert_zoo(seed: int = SEED) -> dict:
    '''The candidate classifier families. Each must be able to produce probabilities.'''
    return {
        "logreg": LogisticRegression(max_iter=2000, class_weight="balanced",
                                     random_state=seed),
        "cnb":    ComplementNB(),
        "sgd":    SGDClassifier(loss="modified_huber", class_weight="balanced",
                                max_iter=2000, random_state=seed),
        # LinearSVC has no predict_proba, so wrap it in Platt scaling
        "svc":    CalibratedClassifierCV(
                      LinearSVC(class_weight="balanced", random_state=seed),
                      cv=3, method="sigmoid"),
    }


def fit_expert(name: str, X, y, seed: int = SEED):
    clf = expert_zoo(seed)[name]
    clf.fit(X, y)
    return clf


def p_fake(clf, X) -> np.ndarray:
    '''P(label == FAKE). Column order varies by classifier, so look it up.'''
    return clf.predict_proba(X)[:, list(clf.classes_).index(FAKE)]

## 5. Metrics

`macro_f1` is the headline (it weights both classes equally, so a model cannot win by
ignoring the minority class). We also record accuracy, precision, recall, ROC-AUC and MCC,
plus the two operationally meaningful error rates: **fake-FNR** (fake articles we let
through) and **fake-FPR** (real articles we wrongly flag).

`tune_threshold` sweeps the decision boundary on validation. Applied to *every* system —
including the baseline — so the comparison stays fair.

In [6]:
def tune_threshold(y, p):
    '''Threshold that maximises macro-F1 on the validation split.'''
    best_t, best_f = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, 91):
        f = f1_score(y, np.where(p >= t, FAKE, REAL), average="macro")
        if f > best_f:
            best_f, best_t = f, t
    return best_t, best_f


def full_metrics(y, p, t) -> dict:
    '''Every metric reported in the paper tables, at a given threshold.'''
    pred = np.where(p >= t, FAKE, REAL)
    cm = confusion_matrix(y, pred, labels=[FAKE, REAL])
    tp, fn, fp, tn = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
    return {
        "macro_f1":  f1_score(y, pred, average="macro"),
        "accuracy":  accuracy_score(y, pred),
        "precision": precision_score(y, pred, pos_label=FAKE, zero_division=0),
        "recall":    recall_score(y, pred, pos_label=FAKE, zero_division=0),
        "roc_auc":   roc_auc_score((y == FAKE).astype(int), p),
        "mcc":       matthews_corrcoef(y, pred),
        "fake_FNR":  fn / max(fn + tp, 1),   # missed fakes
        "fake_FPR":  fp / max(fp + tn, 1),   # false alarms on real news
    }

## 6. S0 — the global baseline

Deliberately a *strong* baseline: it searches the same four families and gets the same
threshold tuning the mixture of experts gets. Beating a weak baseline would prove nothing.

In [7]:
best_g, best_gf, gname = None, -1.0, None
for name in expert_zoo():
    clf = fit_expert(name, X_tr, y_tr)
    f = f1_score(y_va, np.where(p_fake(clf, X_va) >= .5, FAKE, REAL), average="macro")
    print(f"  {name:7s} validation macro-F1 {f:.4f}")
    if f > best_gf:
        best_gf, best_g, gname = f, clf, name

g_va, g_te = p_fake(best_g, X_va), p_fake(best_g, X_te)
t_g, _ = tune_threshold(y_va, g_va)
results = [dict(system=f"S0 global baseline (best={gname})", **full_metrics(y_te, g_te, t_g))]
print(f"\nchosen global family: {gname}, threshold {t_g:.2f}")

  logreg  validation macro-F1 0.7399
  cnb     validation macro-F1 0.6673
  sgd     validation macro-F1 0.7261


  svc     validation macro-F1 0.7318

chosen global family: logreg, threshold 0.51


## 7. Clustering, experts and the soft gate

`build_experts` trains one expert per cluster and lets each cluster **choose its own
family** on its own validation rows. Clusters that are too thin (< 150 training rows) or
single-class fall back to the global model rather than training something unreliable.

`soft_weights` is the gate. Low temperature `T` makes it approach hard routing; high `T`
approaches a uniform average over all experts. `T` is tuned on validation.

In [8]:
def soft_weights(Dx: np.ndarray, centroids: np.ndarray, T: float) -> np.ndarray:
    '''Softmax over negative squared distance to each centroid -> (n, k) weights.'''
    d2 = ((Dx[:, None, :] - centroids[None, :, :]) ** 2).sum(-1)
    z = -d2 / max(T, 1e-6)
    z -= z.max(1, keepdims=True)          # numerical stability
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def build_experts(X_tr, y_tr, c_tr, X_va, y_va, c_va, k, global_clf, min_rows=150):
    '''One expert per cluster; family chosen by validation macro-F1 on that cluster.'''
    experts, chosen = {}, {}
    for c in range(k):
        m_tr, m_va = c_tr == c, c_va == c
        # too small, single-class, or no validation rows -> don't risk a bad specialist
        if m_tr.sum() < min_rows or len(np.unique(y_tr[m_tr])) < 2 or m_va.sum() < 30:
            experts[c], chosen[c] = global_clf, "global(fallback)"
            continue
        best, best_f, bname = None, -1.0, None
        for name in expert_zoo():
            try:
                clf = fit_expert(name, X_tr[m_tr], y_tr[m_tr])
                f = f1_score(y_va[m_va],
                             np.where(p_fake(clf, X_va[m_va]) >= .5, FAKE, REAL),
                             average="macro")
            except Exception:
                continue
            if f > best_f:
                best_f, best, bname = f, clf, name
        experts[c], chosen[c] = (best, bname) if best is not None else (global_clf, "global(fallback)")
    return experts, chosen


def expert_matrix(experts, X, k) -> np.ndarray:
    '''P_fake from every expert for every row -> (n, k).'''
    return np.column_stack([p_fake(experts[c], X) for c in range(k)])

### The hyper-parameter sweep

A grid over cluster count `k`, gate temperature `T` and blend weight `α`, scored on
**validation**. `α = 0` means "experts only" (no global fallback) and `α = 1` would be the
plain global model, so the grid contains the ablation as special cases.

In [9]:
best = None
for k in (4, 6, 8, 10):
    km  = KMeans(k, random_state=SEED, n_init=10).fit(D_tr)
    cen = km.cluster_centers_
    c_tr, c_va, c_te = km.labels_, km.predict(D_va), km.predict(D_te)

    experts, chosen = build_experts(X_tr, y_tr, c_tr, X_va, y_va, c_va, k, best_g)
    E_va, E_te = expert_matrix(experts, X_va, k), expert_matrix(experts, X_te, k)

    for T in (0.02, 0.05, 0.1, 0.25, 0.5):
        W_va, W_te = soft_weights(D_va, cen, T), soft_weights(D_te, cen, T)
        soft_va, soft_te = (W_va * E_va).sum(1), (W_te * E_te).sum(1)
        for a in (0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8):
            p_val = a * g_va + (1 - a) * soft_va
            t, f = tune_threshold(y_va, p_val)
            if best is None or f > best["val_f1"]:
                best = dict(val_f1=f, k=k, T=T, alpha=a, thr=t, chosen=chosen,
                            km=km, experts=experts, c_te=c_te)
    print(f"k={k:2d} done — best validation macro-F1 so far {best['val_f1']:.4f} "
          f"(k={best['k']}, T={best['T']}, alpha={best['alpha']})")

k, T, alpha = best["k"], best["T"], best["alpha"]
print(f"\nSELECTED  k={k}  T={T}  alpha={alpha}")
print("expert family per cluster:", best["chosen"])

k= 4 done — best validation macro-F1 so far 0.7456 (k=4, T=0.25, alpha=0.6)


k= 6 done — best validation macro-F1 so far 0.7456 (k=4, T=0.25, alpha=0.6)


k= 8 done — best validation macro-F1 so far 0.7456 (k=4, T=0.25, alpha=0.6)


k=10 done — best validation macro-F1 so far 0.7456 (k=4, T=0.25, alpha=0.6)

SELECTED  k=4  T=0.25  alpha=0.6
expert family per cluster: {0: 'logreg', 1: 'logreg', 2: 'logreg', 3: 'logreg'}


## 8. Evaluating the ablation on the test split

Six systems, so we can see exactly which architectural change buys what:

| | System | What it isolates |
|---|---|---|
| S1 | hard routing, LogReg experts | Approach 2, reproduced |
| S2 | hard routing, best-family experts | value of heterogeneous experts |
| S3 | soft gating, no blend | value of soft membership |
| S4 | soft gating + global blend | **the full system** |
| S5 | stacked meta-learner | a learned combiner instead of a fixed blend |
| S6 | global ensemble, no clustering | **control** — is it routing, or just averaging? |

In [10]:
km  = best["km"]
cen = km.cluster_centers_
c_tr, c_va, c_te = km.labels_, km.predict(D_va), km.predict(D_te)
experts = best["experts"]

# --- S1: Approach 2 reproduced — one logistic regression per cluster, hard assignment
lr_experts = {}
for c in range(k):
    m = c_tr == c
    lr_experts[c] = (fit_expert("logreg", X_tr[m], y_tr[m])
                     if m.sum() >= 150 and len(np.unique(y_tr[m])) > 1 else best_g)
L_va, L_te = expert_matrix(lr_experts, X_va, k), expert_matrix(lr_experts, X_te, k)
h1_va, h1_te = L_va[np.arange(len(L_va)), c_va], L_te[np.arange(len(L_te)), c_te]
t1, _ = tune_threshold(y_va, h1_va)
results.append(dict(system="S1 hard routing, LogReg experts (Approach 2)",
                    **full_metrics(y_te, h1_te, t1)))

# --- S2: same hard assignment, but each cluster uses its best family
E_va, E_te = expert_matrix(experts, X_va, k), expert_matrix(experts, X_te, k)
h2_va, h2_te = E_va[np.arange(len(E_va)), c_va], E_te[np.arange(len(E_te)), c_te]
t2, _ = tune_threshold(y_va, h2_va)
results.append(dict(system="S2 hard routing, best-family experts",
                    **full_metrics(y_te, h2_te, t2)))

# --- S3: soft gate over experts, but no global fallback
W_va, W_te = soft_weights(D_va, cen, T), soft_weights(D_te, cen, T)
s_va, s_te = (W_va * E_va).sum(1), (W_te * E_te).sum(1)
t3, _ = tune_threshold(y_va, s_va)
results.append(dict(system="S3 soft-gated MoE (no global blend)",
                    **full_metrics(y_te, s_te, t3)))

# --- S4: the full system
p_va = alpha * g_va + (1 - alpha) * s_va
p_te = alpha * g_te + (1 - alpha) * s_te
t4, _ = tune_threshold(y_va, p_va)
results.append(dict(system=f"S4 soft MoE + global blend (k={k}, T={T}, a={alpha})",
                    **full_metrics(y_te, p_te, t4)))

# --- S5: let a meta-model learn the combination instead of fixing it
M_va = np.column_stack([g_va, E_va, W_va])
M_te = np.column_stack([g_te, E_te, W_te])
meta = LogisticRegression(max_iter=2000, class_weight="balanced",
                          random_state=SEED).fit(M_va, y_va)
m_va, m_te = p_fake(meta, M_va), p_fake(meta, M_te)
t5, _ = tune_threshold(y_va, m_va)
results.append(dict(system="S5 stacked meta-learner", **full_metrics(y_te, m_te, t5)))

# --- S6 CONTROL: average the four GLOBAL models, no clustering at all.
# If this matches S4, the gain was ensembling rather than routing.
gl = {n: fit_expert(n, X_tr, y_tr) for n in expert_zoo()}
e_va = np.column_stack([p_fake(c, X_va) for c in gl.values()]).mean(1)
e_te = np.column_stack([p_fake(c, X_te) for c in gl.values()]).mean(1)
t6, _ = tune_threshold(y_va, e_va)
results.append(dict(system="S6 CONTROL global ensemble (no clustering)",
                    **full_metrics(y_te, e_te, t6)))

res = pd.DataFrame(results).round(4)
res

,system,macro_f1,accuracy,precision,recall,roc_auc,mcc,fake_FNR,fake_FPR
0,S0 global baseline (best=logreg),0.7657,0.7679,0.7593,0.8194,0.8330,0.5341,0.1806,0.2896
1,"S1 hard routing, LogReg experts (Approach 2)",0.7494,0.7511,0.7508,0.7900,0.8349,0.4999,0.2100,0.2923
2,"S2 hard routing, best-family experts",0.7494,0.7511,0.7508,0.7900,0.8349,0.4999,0.2100,0.2923
3,S3 soft-gated MoE (no global blend),0.7620,0.7649,0.7507,0.8292,0.8391,0.5288,0.1708,0.3069
4,"S4 soft MoE + global blend (k=4, T=0.25, a=0.6)",0.7665,0.7692,0.7556,0.8309,0.8388,0.5373,0.1691,0.2996
5,S5 stacked meta-learner,0.7647,0.7670,0.7570,0.8219,0.8386,0.5326,0.1781,0.2942
6,S6 CONTROL global ensemble (no clustering),0.7600,0.7631,0.7471,0.8325,0.8343,0.5256,0.1675,0.3142


## 9. Reading the result

Expect to see three things.

**The architectural fixes work.** Each step recovers part of what hard routing gave away:
hard LogReg routing is the worst row, heterogeneous experts recover some, soft gating a
little more, and the global blend closes the gap completely.

**But routing still does not beat a strong global model.** S4 exceeds S0 by about
+0.0008 (FineFake) and +0.0004 (WELFake) — inside run-to-run noise. The honest claim is
that the mixture of experts **matches** the global model.

**The control explains why that is still interesting.** S6 — the same four classifiers
ensembled without any clustering — scores *below* the single best global model, so S4's
small edge is not merely "ensembles help".

The real accuracy gain over this project's earlier numbers (≈ +1.3 macro-F1 points on both
corpora) comes from **choosing the right classifier family and tuning the decision
threshold**, not from clustering.

In [11]:
out = REPO / "results"
out.mkdir(exist_ok=True)
res.insert(0, "dataset", DATASET)
res.to_csv(out / f"moe_{DATASET}.csv", index=False)
print("written:", out / f"moe_{DATASET}.csv")

written: /home/claude/repo_fresh/results/moe_finefake.csv


## 10. Using the model on a new article

The end-to-end inference path, exactly as the architecture diagram describes it:
vectorise → project → gate → weight the experts → blend the global model back in →
threshold. It returns the probability, the verdict, and the gate weights, so the routing
decision is inspectable rather than hidden.

In [12]:
def predict(texts, return_weights: bool = True) -> pd.DataFrame:
    '''Score raw article text with the full Approach-3 pipeline.'''
    Xs = vec.transform(texts)                  # sparse features for the classifiers
    Ds = svd.transform(Xs)                     # dense features for the gate
    W  = soft_weights(Ds, cen, T)              # (n, k) soft cluster membership
    P  = expert_matrix(experts, Xs, k)         # (n, k) expert probabilities
    p  = alpha * p_fake(best_g, Xs) + (1 - alpha) * (W * P).sum(1)

    out = pd.DataFrame({
        "p_fake":  p.round(4),
        "verdict": np.where(p >= t4, "FAKE", "REAL"),
        "top_cluster": W.argmax(1),
    })
    if return_weights:
        out["gate_weights"] = [np.round(w, 3).tolist() for w in W]
    return out


demo = te["text"].head(5).tolist()
preview = predict(demo)
preview.insert(0, "true", np.where(y_te[:5] == FAKE, "FAKE", "REAL"))
preview.insert(1, "snippet", [t[:70].replace("\n", " ") + "…" for t in demo])
preview

,true,snippet,p_fake,verdict,top_cluster,gate_weights
0,FAKE,Tom Brady was recorded telling his teammates ...,0.5856,FAKE,3,"[0.097, 0.335, 0.094, 0.474]"
1,REAL,"On Oct. 1, 2020, the re-election campaign of ...",0.6183,FAKE,3,"[0.095, 0.228, 0.094, 0.583]"
2,FAKE,A photograph that went viral in June 2023 aut...,0.8153,FAKE,3,"[0.136, 0.152, 0.102, 0.61]"
3,REAL,A video filmed at the microscopic level shows...,0.5258,FAKE,3,"[0.088, 0.302, 0.094, 0.515]"
4,REAL,"U.S. President Joe Biden was recorded on a ""h...",0.7261,FAKE,3,"[0.123, 0.21, 0.1, 0.567]"


## 11. Limitations

1. **Single seed.** Treat differences below ~0.01 macro-F1 as ties. Bootstrap confidence
   intervals over several seeds are the obvious next step.
2. **Validation reuse.** The grid sweeps `k × T × α × threshold` on one validation split,
   so the selected configuration is mildly optimistic; nested cross-validation would tighten it.
3. **In-distribution only.** Separate transfer experiments showed these scores fall sharply
   on a different corpus — a benchmark score is corpus fit, not detection ability.
4. **Shortcut inheritance.** Clusters are built from the same TF-IDF features the
   classifiers use, so routing redistributes whatever surface artifacts the features carry.
   A genuine specialisation gain likely needs a routing signal *independent* of the
   classifier's own features.